# Data Loading: MCO vs MIA Analysis

Reads flight data from Dropbox and filters for MCO/MIA airports.

**Method:** Chunk processing - reads compressed .bz2 files directly and filters simultaneously.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os

print(f"pandas {pd.__version__}")

## Configuration

In [ ]:
# Target airports
AIRPORTS = ['MCO', 'MIA']

# Years to process
YEARS = [2004, 2005, 2006, 2007, 2008]

# Output directory
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Airports: {', '.join(AIRPORTS)}")
print(f"Years: {YEARS[0]}-{YEARS[-1]}")
print(f"Output: {OUTPUT_DIR}")

## Load Dropbox URLs from secrets.txt

In [ ]:
# Read Dropbox URLs from secrets.txt (not committed to GitHub)
with open('secrets.txt', 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

# Map URLs to years
DROPBOX_URLS = dict(zip(YEARS, urls))

print(f"Loaded {len(DROPBOX_URLS)} URLs from secrets.txt")

## Load and Filter Function

In [ ]:
def load_and_filter(url, year, output_dir, airports, chunksize=100000):
    """
    Load .bz2 CSV from Dropbox and filter for specified airports in chunks.
    
    Parameters:
    - url: Dropbox direct download URL
    - year: Year being processed
    - output_dir: Directory to save filtered data
    - airports: List of airport codes to filter
    - chunksize: Rows per chunk (default 100000)
    
    Returns:
    - DataFrame with filtered data
    """
    output_file = f'{output_dir}/mco_mia_{year}.csv'
    
    print(f"\nProcessing {year}")
    
    chunk_list = []
    total_rows = 0
    filtered_rows = 0
    
    try:
        # Read compressed file in chunks (pandas handles .bz2 automatically)
        for i, chunk in enumerate(pd.read_csv(url, compression='bz2', chunksize=chunksize, low_memory=False), 1):
            total_rows += len(chunk)
            
            # Filter for target airports
            filtered = chunk[
                (chunk['Origin'].isin(airports)) | 
                (chunk['Dest'].isin(airports))
            ]
            
            if len(filtered) > 0:
                chunk_list.append(filtered)
                filtered_rows += len(filtered)
            
            # Progress every 10 chunks
            if i % 10 == 0:
                print(f"  Processed {total_rows:,} rows, kept {filtered_rows:,}")
        
        # Combine and save
        df = pd.concat(chunk_list, ignore_index=True)
        df.to_csv(output_file, index=False)
        
        reduction = (1 - filtered_rows / total_rows) * 100
        print(f"Complete: {filtered_rows:,} / {total_rows:,} rows ({reduction:.1f}% reduction)")
        print(f"Saved to: {output_file}")
        
        return df
        
    except Exception as e:
        print(f"Error: {e}")
        return None

## Process All Years

In [ ]:
for year in YEARS:
    url = DROPBOX_URLS.get(year)
    if url:
        load_and_filter(url, year, OUTPUT_DIR, AIRPORTS)
    else:
        print(f"Skipping {year}: URL not found")

## Combine All Years

In [ ]:
dfs = []

for year in YEARS:
    filepath = f'{OUTPUT_DIR}/mco_mia_{year}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        dfs.append(df)
        print(f"{year}: {len(df):,} rows")
    else:
        print(f"{year}: file not found")

if dfs:
    # Combine all years
    df_combined = pd.concat(dfs, ignore_index=True)
    
    # Save combined file
    output_path = f'{OUTPUT_DIR}/mco_mia_2004_2008.csv'
    df_combined.to_csv(output_path, index=False)
    
    print(f"\nCombined: {len(df_combined):,} total rows")
    print(f"Saved to: {output_path}")
    
    # Preview
    display(df_combined.head())
    
    # Summary
    print(f"\nColumns: {len(df_combined.columns)}")
    print(f"File size: {os.path.getsize(output_path) / 1024**2:.1f} MB")
else:
    print("\nNo data files found")

## Next Step

Proceed to **02_data_cleaning.ipynb**